<a href="https://colab.research.google.com/github/HafizaNoorUlSaba/langgraph/blob/main/prompt_chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Libraries


In [8]:
!pip install langchain langgraph typing langchain-google-genai langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.3 MB/s eta 0:00:00


In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [16]:
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets and set it as an environment variable
# Ensure you have a secret named 'GROQ_API_KEY' in Colab secrets.
groq_api_key_from_secrets = userdata.get('Groq_Api_Key')

if groq_api_key_from_secrets:
    os.environ['GROQ_API_KEY'] = groq_api_key_from_secrets
    print("GROQ_API_KEY loaded from Colab secrets.")
else:
    print("Warning: GROQ_API_KEY secret not found in Colab secrets. Please ensure it's set correctly.")

GROQ_API_KEY loaded from Colab secrets.


define llm

In [62]:
llm = ChatGroq(
    groq_api_key=os.getenv("Groq_Api_Key"),
    model_name="openai/gpt-oss-120b" # Corrected to a supported Groq model
)
model = llm # Assign the initialized LLM to the 'model' variable to match the existing functions

define state

In [37]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

Creating Nodes

In [38]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [39]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

Define and Compile Graph

In [40]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

Execute the graph

In [43]:
intial_state = {'title': 'Rise of agentic ai'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'Rise of agentic ai', 'outline': '**Blog Title:** *The Rise of Agentic AI – How Self‑Directed Machines Are Redefining Intelligence, Work, and Society*  \n\n**Target Audience:** Tech‑savvy professionals, AI researchers, policy makers, futurists, and curious readers who want a deep yet accessible look at the emerging class of “agentic” artificial intelligence.\n\n---\n\n## 1. Introduction (≈300‑400\u202fwords)\n\n| Element | What to Cover |\n|---------|----------------|\n| **Hook** | A vivid, relatable vignette – e.g., a personal assistant that not only schedules meetings *but* negotiates a better contract on your behalf. |\n| **Definition teaser** | Introduce “agentic AI” as AI systems that can set goals, make decisions, and act autonomously in open‑ended environments. |\n| **Why it matters now** | Link to recent breakthroughs (large language models, reinforcement‑learning‑from‑human‑feedback, multimodal agents) and real‑world deployments (autonomous trading bots, AI‑driven su

In [44]:
print(final_state['outline'])

**Blog Title:** *The Rise of Agentic AI – How Self‑Directed Machines Are Redefining Intelligence, Work, and Society*  

**Target Audience:** Tech‑savvy professionals, AI researchers, policy makers, futurists, and curious readers who want a deep yet accessible look at the emerging class of “agentic” artificial intelligence.

---

## 1. Introduction (≈300‑400 words)

| Element | What to Cover |
|---------|----------------|
| **Hook** | A vivid, relatable vignette – e.g., a personal assistant that not only schedules meetings *but* negotiates a better contract on your behalf. |
| **Definition teaser** | Introduce “agentic AI” as AI systems that can set goals, make decisions, and act autonomously in open‑ended environments. |
| **Why it matters now** | Link to recent breakthroughs (large language models, reinforcement‑learning‑from‑human‑feedback, multimodal agents) and real‑world deployments (autonomous trading bots, AI‑driven supply‑chain managers, digital twins). |
| **Road‑map of the po

In [46]:
print(final_state['content'])

**The Rise of Agentic AI – How Self‑Directed Machines Are Redefining Intelligence, Work, and Society**  
*Target audience: tech‑savvy professionals, AI researchers, policy‑makers, futurists, and curious readers*  

---  

## 1. Introduction  *(≈350 words)*  

**Hook – a day in the life of a “hyper‑assistant”**  

Imagine waking up to a notification that your calendar has already been rearranged: the 10 am client call has been moved to 2 pm, a better‑priced contract for the new software license has just been negotiated, and a draft of the revised proposal is waiting in your inbox with a polite note, “I’ve added the client’s latest budget constraints – let me know if you’d like any tweaks.” You didn’t type a single email, you didn’t open a spreadsheet, and you certainly didn’t spend an hour on a negotiation call. An AI **agent** did it all, reasoning about your goals, pulling in real‑time market data, and executing actions across several SaaS tools—all before you even brushed your teeth.